In [6]:
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from shapely.geometry import mapping, box
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import pandas as pd
import os
from tqdm import tqdm
import joblib
import logging
import time
import warnings
warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

# Set up logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S')
logger = logging.getLogger(__name__)

# Define class labels
CLASS_MAPPING = {
    'lake': 1,
    'river': 2,
    'land': 0
}

def load_polygons(shapefile_path, class_field='land_type'):
    """
    Load polygons from a shapefile with class labels
    
    Parameters:
    shapefile_path: Path to the shapefile containing polygons
    class_field: Field in the shapefile that contains the class labels
    
    Returns:
    GeoDataFrame containing the polygons with class information
    """
    gdf = gpd.read_file(shapefile_path)
    
    # Print columns and unique class values for debugging
    print(f"Shapefile columns: {list(gdf.columns)}")
    print(f"Unique values in {class_field}: {np.unique(gdf[class_field])}")
    print(f"Shapefile CRS: {gdf.crs}")
    
    # Ensure the GeoDataFrame has the specified class field
    if class_field not in gdf.columns:
        raise ValueError(f"Shapefile must contain a '{class_field}' column with values: lake, river, or land")
    
    # Convert class names to numeric labels
    gdf['label'] = gdf[class_field].map(CLASS_MAPPING)
    
    # Check if any classes couldn't be mapped
    if gdf['label'].isna().any():
        unknown_classes = gdf[gdf['label'].isna()][class_field].unique()
        logger.warning(f"Unknown classes found in shapefile: {unknown_classes}")
        # Remove rows with unknown classes
        gdf = gdf.dropna(subset=['label'])
        
    # Convert label to integer
    gdf['label'] = gdf['label'].astype(int)
    
    # Report class distribution
    class_counts = gdf[class_field].value_counts().to_dict()
    print("Polygon class distribution:")
    for class_name, count in class_counts.items():
        print(f"  {class_name}: {count} polygons")
    
    return gdf

def load_band(band_path):
    """
    Load a single band from a raster file
    
    Parameters:
    band_path: Path to the band file
    
    Returns:
    Dictionary with band data, transform, CRS, and profile
    """
    if not os.path.exists(band_path):
        raise FileNotFoundError(f"Band file not found: {band_path}")
        
    try:
        with rasterio.open(band_path) as src:
            band_data = {
                'data': src.read(1),  # Read the first band
                'transform': src.transform,
                'crs': src.crs,
                'profile': src.profile,
                'width': src.width,
                'height': src.height,
                'bounds': src.bounds
            }
        print(f"Successfully loaded band from {os.path.basename(band_path)}")
        return band_data
    except Exception as e:
        raise RuntimeError(f"Error loading band {band_path}: {str(e)}")

def load_bands(band_paths):
    """
    Load all bands from a dictionary of band paths
    
    Parameters:
    band_paths: Dictionary with keys 'blue', 'green', 'red', 'nir' and values as paths
    
    Returns:
    Dictionary with loaded band data
    """
    bands = {}
    for band_name, band_path in band_paths.items():
        try:
            bands[band_name] = load_band(band_path)
            print(f"Successfully loaded {band_name} band from {os.path.basename(band_path)}")
        except Exception as e:
            logger.error(f"Failed to load {band_name} band: {str(e)}")
            raise
    
    return bands

def process_single_image(image_idx, band_paths, polygons_gdf, max_samples_per_polygon=10000, all_touched=True, min_valid_ratio=0.1):
    """
    Process a single image with all overlapping polygons - USING ONLY BASE SPECTRAL BANDS
    """
    # Load bands
    try:
        print(f"Processing image set {image_idx+1}")
        bands = load_bands(band_paths)
        
        # Use blue band for reference
        reference_band = bands['blue']
        
        # Get the bounds of the image
        left, bottom, right, top = rasterio.transform.array_bounds(
            reference_band['height'], 
            reference_band['width'], 
            reference_band['transform']
        )
        
        # Create a box for the image bounds
        image_box = box(left, bottom, right, top)
        
        # Print debug information
        print(f"Image bounds: {left}, {bottom}, {right}, {top}")
        
        # If CRS is different, reproject polygons to match image CRS
        if reference_band['crs'] != polygons_gdf.crs:
            print(f"Reprojecting polygons from {polygons_gdf.crs} to {reference_band['crs']}")
            polygons_in_image_crs = polygons_gdf.to_crs(reference_band['crs'])
        else:
            polygons_in_image_crs = polygons_gdf
        
        # Find polygons that intersect with the image
        polygons_in_image = polygons_in_image_crs[polygons_in_image_crs.intersects(image_box)]
        print(f"Found {len(polygons_in_image)} overlapping polygons")
        
        if len(polygons_in_image) == 0:
            print("No overlapping polygons found for this image")
            return None, None, None
        
        X_list = []
        y_list = []
        polygon_ids_list = []
        
        # Process each polygon
        for idx, row in tqdm(polygons_in_image.iterrows(), 
                           total=len(polygons_in_image), 
                           desc=f"Extracting from image set {image_idx+1}"):
            
            try:
                # Create a mask for the polygon
                with rasterio.Env():
                    # Create a 2D boolean mask for the polygon
                    mask_geom = rasterio.features.geometry_mask(
                        [mapping(row.geometry)],
                        out_shape=(reference_band['height'], reference_band['width']),
                        transform=reference_band['transform'],
                        all_touched=all_touched,
                        invert=True
                    )
                
                # If no pixels in polygon, skip
                pixel_count = np.sum(mask_geom)
                if pixel_count == 0:
                    print(f"  Skipping polygon {idx} (class: {row['label']}) - no pixels in mask")
                    continue
                
                # Prepare feature arrays - ONLY use spectral bands, NO INDICES
                feature_arrays = []
                
                # Add spectral bands
                for band_name in ['blue', 'green', 'red', 'nir']:
                    if band_name in bands:
                        try:
                            band_data = bands[band_name]['data']
                            masked_data = band_data[mask_geom]
                            feature_arrays.append(masked_data)
                        except Exception as e:
                            print(f"  Error extracting {band_name} band: {e}")
                
                # If we don't have enough features, skip this polygon
                if len(feature_arrays) < 4:  # Require all 4 base bands
                    print(f"  Skipping polygon {idx} - missing some spectral bands")
                    continue
                    
                try:
                    # Check that all arrays have the same length
                    lengths = [arr.shape[0] for arr in feature_arrays]
                    if len(set(lengths)) > 1:
                        print(f"  Error: Feature arrays have different lengths: {lengths}")
                        continue
                        
                    # Column stack features
                    X = np.column_stack(feature_arrays)
                    y = np.full(X.shape[0], row['label'])
                    polygon_id = np.full(X.shape[0], idx)
                    
                    # Sample if too large
                    if X.shape[0] > max_samples_per_polygon:
                        sample_indices = np.random.choice(X.shape[0], max_samples_per_polygon, replace=False)
                        X = X[sample_indices]
                        y = y[sample_indices]
                        polygon_id = polygon_id[sample_indices]
                    
                    X_list.append(X)
                    y_list.append(y)
                    polygon_ids_list.append(polygon_id)
                    
                    # Print success for this polygon
                    print(f"  Successfully extracted {X.shape[0]} samples from polygon {idx}")
                
                except Exception as e:
                    print(f"  Error processing features for polygon {idx}: {e}")
                    continue
                
            except Exception as e:
                print(f"  Error processing polygon {idx}: {e}")
                continue
        
        # Combine all samples
        if len(X_list) > 0:
            try:
                X_all = np.vstack(X_list)
                y_all = np.hstack(y_list)
                polygon_ids_all = np.hstack(polygon_ids_list)
                print(f"Extracted {X_all.shape[0]} pixels with {X_all.shape[1]} features")
                return X_all, y_all, polygon_ids_all
            except Exception as e:
                print(f"Error combining samples: {e}")
                return None, None, None
        else:
            print("No features extracted from any polygons")
            return None, None, None
            
    except Exception as e:
        print(f"Error processing image {image_idx}: {e}")
        return None, None, None

def create_training_dataset(band_paths_list, polygons_gdf, max_samples_per_polygon=10000, all_touched=True, min_valid_ratio=0.1):
    """
    Create training dataset by processing all images - using only base spectral bands
    
    Returns:
    X: Feature array
    y: Labels array
    polygon_ids: Array of polygon IDs for each sample
    feature_names: List of feature names
    """
    start_time = time.time()
    
    X_list = []
    y_list = []
    polygon_ids_list = []
    
    success_count = 0
    
    for i, band_paths in enumerate(band_paths_list):
        print(f"\n--- Processing image set {i+1}/{len(band_paths_list)} ---")
        
        X, y, polygon_ids = process_single_image(
            i, 
            band_paths, 
            polygons_gdf, 
            max_samples_per_polygon=max_samples_per_polygon,
            all_touched=all_touched,
            min_valid_ratio=min_valid_ratio
        )
        
        if X is not None and y is not None and X.shape[0] > 0:
            X_list.append(X)
            y_list.append(y)
            polygon_ids_list.append(polygon_ids)
            success_count += 1
            print(f"Image {i+1} processing successful: extracted {X.shape[0]} samples")
        else:
            print(f"Image {i+1} processing failed or yielded no samples")
    
    print(f"Successfully processed {success_count} out of {len(band_paths_list)} images")
    
    # Combine all samples - only if we have at least one successful image
    if len(X_list) > 0:
        try:
            X = np.vstack(X_list)
            y = np.hstack(y_list)
            polygon_ids = np.hstack(polygon_ids_list)
            
            # Define feature names - always just the base spectral bands
            feature_names = ['Blue', 'Green', 'Red', 'NIR']
            
            # Log dataset statistics
            class_counts = {name: np.sum(y == label) for name, label in CLASS_MAPPING.items()}
            total_samples = len(y)
            
            print("\nTraining dataset created:")
            print(f"Total samples: {total_samples}")
            print(f"Features: {X.shape[1]} ({', '.join(feature_names)})")
            print("Class distribution:")
            for class_name, count in class_counts.items():
                percentage = (count / total_samples) * 100
                print(f"  {class_name}: {count} samples ({percentage:.1f}%)")
            
            print(f"Dataset creation took {time.time() - start_time:.1f} seconds")
            
            return X, y, polygon_ids, feature_names
        except Exception as e:
            print(f"Error creating final dataset: {e}")
            raise ValueError(f"Failed to create dataset: {e}")
    else:
        print("No valid data could be extracted from any images.")
        print("Check your input data and make sure polygons overlap with images.")
        
        # Add more troubleshooting info
        for i, band_paths in enumerate(band_paths_list):
            print(f"\nImage {i+1} paths:")
            for band, path in band_paths.items():
                print(f"  {band}: {os.path.basename(path)}")
                if not os.path.exists(path):
                    print(f"    WARNING: This file does not exist!")
        
        raise ValueError("No training data could be extracted from the images")

def train_water_model(X, y, polygon_ids, model_type='rf', max_samples=500000):
    """
    Train a model with sampling for large datasets - with improved evaluation
    
    Parameters:
    X: Feature array
    y: Labels array
    polygon_ids: Array of polygon IDs for each sample
    model_type: 'rf' for Random Forest or 'svm' for SVM
    max_samples: Maximum number of samples to use for training
    
    Returns:
    Trained model and metrics
    """
    # Sample data if it's too large
    if X.shape[0] > max_samples:
        # Stratified sampling to maintain class distribution
        from sklearn.model_selection import StratifiedShuffleSplit
        sss = StratifiedShuffleSplit(n_splits=1, train_size=max_samples, random_state=42)
        for train_index, _ in sss.split(X, y):
            X_sampled = X[train_index]
            y_sampled = y[train_index]
            polygon_ids_sampled = polygon_ids[train_index]
        logger.info(f"Sampled {max_samples} out of {X.shape[0]} samples for training")
    else:
        X_sampled = X
        y_sampled = y
        polygon_ids_sampled = polygon_ids
    
    # Get unique polygon IDs
    unique_polygons = np.unique(polygon_ids_sampled)
    print(f"Total unique polygons: {len(unique_polygons)}")
    
    # Split polygons into train and test sets
    from sklearn.model_selection import train_test_split
    train_polygons, test_polygons = train_test_split(
        unique_polygons, test_size=0.3, random_state=42
    )
    
    # Create masks for train and test pixels based on polygon IDs
    train_mask = np.isin(polygon_ids_sampled, train_polygons)
    test_mask = np.isin(polygon_ids_sampled, test_polygons)
    
    # Split data using the masks
    X_train, X_test = X_sampled[train_mask], X_sampled[test_mask]
    y_train, y_test = y_sampled[train_mask], y_sampled[test_mask]
    
    print(f"Training set size: {X_train.shape[0]} samples from {len(train_polygons)} polygons")
    print(f"Testing set size: {X_test.shape[0]} samples from {len(test_polygons)} polygons")
    
    # Check for class imbalance in train and test sets
    train_class_counts = {}
    test_class_counts = {}
    for class_name, class_id in CLASS_MAPPING.items():
        train_count = np.sum(y_train == class_id)
        train_class_counts[class_name] = train_count
        test_count = np.sum(y_test == class_id)
        test_class_counts[class_name] = test_count
    
    print("Class distribution in training set:")
    for class_name, count in train_class_counts.items():
        print(f"  {class_name}: {count} samples ({count/len(y_train)*100:.1f}%)")
    
    print("Class distribution in test set:")
    for class_name, count in test_class_counts.items():
        print(f"  {class_name}: {count} samples ({count/len(y_test)*100:.1f}%)")
    
    # Check for spatial autocorrelation (optional)
    spatial_autocorrelation_ratio = check_spatial_autocorrelation(X_sampled, y_sampled, polygon_ids_sampled)
    if spatial_autocorrelation_ratio is not None and spatial_autocorrelation_ratio > 0.8:
        print("WARNING: High spatial autocorrelation detected. Model metrics may be overly optimistic.")
    
    if model_type == 'rf':
        # Train Random Forest model with parameters to reduce overfitting
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,        # Reduced to prevent overfitting
            min_samples_split=25, # Increased to prevent overfitting
            min_samples_leaf=15,  # Increased to prevent overfitting
            class_weight='balanced',
            max_features='sqrt',  
            oob_score=True,       
            random_state=42,
            n_jobs=-1
        )
    else:
        # Train SVM model
        from sklearn.svm import SVC
        model = SVC(
            kernel='rbf',
            C=10,
            gamma='scale',
            class_weight='balanced',
            probability=True,
            random_state=42
        )
    
    # Optional: Feature selection to reduce overfitting
    print("Performing feature selection...")
    from sklearn.feature_selection import SelectFromModel
    from sklearn.tree import DecisionTreeClassifier
    
    selector = SelectFromModel(
        DecisionTreeClassifier(max_depth=4, random_state=42),
        threshold='median'
    )
    selector.fit(X_train, y_train)
    
    # Transform data with selected features
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)
    
    # Get selected feature indices
    selected_features = selector.get_support()
    print(f"Selected {np.sum(selected_features)} out of {len(selected_features)} features")
    
    # Train the model
    print(f"Training {model_type} model...")
    model.fit(X_train_selected, y_train)
    
    # For Random Forest, print out-of-bag error estimate
    if model_type == 'rf':
        print(f"Out-of-bag score: {model.oob_score_:.3f}")
    
    # Evaluate on test set
    y_pred = model.predict(X_test_selected)
    
    # Calculate metrics
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average=None)
    recall = recall_score(y_test, y_pred, average=None)
    f1 = f1_score(y_test, y_pred, average=None)
    
    # Full classification report
    report = classification_report(y_test, y_pred, target_names=list(CLASS_MAPPING.keys()))
    print("\nClassification Report:")
    print(report)
    
    cm = confusion_matrix(y_test, y_pred)
    
    # Print metrics by class
    print("\nMetrics by class:")
    for i, class_name in enumerate(CLASS_MAPPING.keys()):
        print(f"{class_name}: Precision={precision[i]:.3f}, Recall={recall[i]:.3f}, F1={f1[i]:.3f}")
    
    # Analyze feature importance for Random Forest
    if model_type == 'rf':
        feature_importance = model.feature_importances_
    else:
        feature_importance = None
    
    return {
        'model': model,
        'selector': selector,  # Include selector for prediction
        'report': report,
        'confusion_matrix': cm,
        'feature_importance': feature_importance,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

def apply_model_to_image(model_results, band_paths, output_path, model_type='rf', probabilities=False, batch_size=1000000):
    """
    Apply the trained model to a new satellite image - USING ONLY BASE SPECTRAL BANDS
    
    Parameters:
    model_results: Dictionary with trained models and metrics
    band_paths: Dictionary with paths to the bands
    output_path: Path to save the classification result
    model_type: 'rf' for Random Forest or 'svm' for SVM
    probabilities: Whether to save probability maps
    batch_size: Size of batches to process to reduce memory usage
    
    Returns:
    Classification array
    """
    # Get the model and feature selector
    model = model_results[model_type]['model']
    selector = model_results[model_type].get('selector')  # May be None if no feature selection was used
    
    # Load bands
    bands = load_bands(band_paths)
    
    # Get reference band
    reference_band = bands['blue']
    height, width = reference_band['height'], reference_band['width']
    
    # Create a mask for valid pixels
    valid_mask = np.ones((height, width), dtype=bool)
    for band_name in bands:
        band_data = bands[band_name]['data']
        valid_mask &= (band_data > 0) & (band_data < 65535) & (~np.isnan(band_data))
    
    # Count valid pixels
    valid_count = np.sum(valid_mask)
    logger.info(f"Valid pixels: {valid_count} out of {height*width} ({valid_count/(height*width)*100:.2f}%)")
    
    # If there are no valid pixels, return
    if valid_count == 0:
        logger.error("No valid pixels found in the image")
        return None
    
    # Prepare feature arrays - ONLY USE BASE SPECTRAL BANDS
    feature_arrays = []
    
    # Add spectral bands - in the exact same order used for training
    for band_name in ['blue', 'green', 'red', 'nir']:
        if band_name in bands:
            band_data = bands[band_name]['data'][valid_mask]
            feature_arrays.append(band_data)
    
    # Stack features
    X = np.column_stack(feature_arrays)
    
    # Report feature count
    print(f"Prediction features: {X.shape[1]} (blue, green, red, nir)")
    
    # Apply feature selection if it was used
    if selector is not None:
        X = selector.transform(X)
        print(f"After selection: {X.shape[1]} features")
    
    # Process in batches
    result = np.zeros(valid_count, dtype=np.uint8)
    
    # Process batches
    for i in range(0, valid_count, batch_size):
        end = min(i + batch_size, valid_count)
        result[i:end] = model.predict(X[i:end])
    
    # Reshape result to the original image dimensions
    classification = np.zeros((height, width), dtype=np.uint8)
    classification[valid_mask] = result
    
    # Save classification
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=rasterio.uint8,
        crs=reference_band['crs'],
        transform=reference_band['transform'],
        nodata=255
    ) as dst:
        dst.write(classification, 1)
    
    # Save probability maps if requested
    if probabilities:
        # Process in batches
        prob_results = []
        for i in range(0, valid_count, batch_size):
            end = min(i + batch_size, valid_count)
            prob_results.append(model.predict_proba(X[i:end]))
        
        # Combine batches
        all_probs = np.vstack(prob_results)
        
        # For each class
        class_mapping_inv = {v: k for k, v in CLASS_MAPPING.items()}
        for class_id in range(len(class_mapping_inv)):
            class_name = class_mapping_inv[class_id]
            # Create probability array
            prob_array = np.zeros((height, width), dtype=np.float32)
            prob_array[valid_mask] = all_probs[:, class_id]
            
            # Save to file
            prob_path = output_path.replace('.tif', f'_{class_name}_prob.tif')
            with rasterio.open(
                prob_path,
                'w',
                driver='GTiff',
                height=height,
                width=width,
                count=1,
                dtype=rasterio.float32,
                crs=reference_band['crs'],
                transform=reference_band['transform'],
                nodata=-1
            ) as dst:
                dst.write(prob_array, 1)
    
    # Create a better visualization
    vis_path = output_path.replace('.tif', '_viz.png')
    create_enhanced_visualization(classification, bands, vis_path)
    
    return classification

def check_spatial_autocorrelation(X, y, polygon_ids, distance_threshold=100):
    """
    Check for spatial autocorrelation in the results
    
    This helps detect if model performance is artificially high due to
    spatial dependency in nearby pixels
    """
    from sklearn.metrics.pairwise import euclidean_distances
    import random
    
    # Sample a smaller subset for computational efficiency
    max_samples = 1000
    if len(X) > max_samples:
        indices = random.sample(range(len(X)), max_samples)
        X_sample = X[indices]
        y_sample = y[indices]
        polygon_ids_sample = polygon_ids[indices]
    else:
        X_sample = X
        y_sample = y
        polygon_ids_sample = polygon_ids
    
    # Calculate pairwise distances
    distances = euclidean_distances(X_sample)
    
    # Find nearby pixel pairs
    nearby_pairs = np.where(distances < distance_threshold)
    
    # Count how many nearby pairs have the same class
    same_class_count = np.sum(y_sample[nearby_pairs[0]] == y_sample[nearby_pairs[1]])
    total_pairs = len(nearby_pairs[0])
    
    # Calculate the ratio
    if total_pairs > 0:
        same_class_ratio = same_class_count / total_pairs
        print(f"Spatial autocorrelation check:")
        print(f"  {same_class_count} out of {total_pairs} nearby pixel pairs have the same class")
        print(f"  Ratio: {same_class_ratio:.3f}")
        print(f"  A high ratio (>0.8) suggests strong spatial autocorrelation")
        
        return same_class_ratio
    else:
        print("No nearby pixel pairs found for autocorrelation check")
        return None

def create_enhanced_visualization(classification, bands, output_path):
    """
    Create a better visualization with RGB background and class overlay
    
    Parameters:
    classification: Classification array
    bands: Dictionary with loaded band data
    output_path: Path to save the visualization
    """
    # Create RGB background
    height, width = classification.shape
    rgb = np.zeros((height, width, 3), dtype=np.float32)
    
    # Normalize RGB bands for visualization
    band_order = ['red', 'green', 'blue']
    for i, band_name in enumerate(band_order):
        if band_name in bands:
            band_data = bands[band_name]['data']
            # Simple percentile-based normalization
            p2 = np.percentile(band_data, 2)
            p98 = np.percentile(band_data, 98)
            normalized = np.clip((band_data - p2) / (p98 - p2), 0, 1)
            rgb[:, :, i] = normalized
    
    # Create colored overlay for water classes
    overlay = np.zeros((height, width, 4), dtype=np.float32)
    
    # Define colors with alpha for transparency
    colors = {
        0: [0, 0, 0, 0],        # Land - transparent
        1: [0, 0.5, 0.8, 0.7],  # Lake - blue with alpha
        2: [0, 0.3, 0.6, 0.7]   # River - dark blue with alpha
    }
    
    # Apply colors to overlay
    for class_val, color in colors.items():
        mask = classification == class_val
        for i in range(4):  # RGBA
            overlay[:, :, i][mask] = color[i]
    
    # Create figure
    plt.figure(figsize=(12, 12))
    
    # Plot RGB background
    plt.imshow(rgb)
    
    # Plot water class overlay
    rgba = np.zeros((overlay.shape[0], overlay.shape[1], 4))
    for i in range(4):
        rgba[:, :, i] = overlay[:, :, i]
    plt.imshow(rgba)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='none', edgecolor='none', label='Land'),
        Patch(facecolor=(0, 0.5, 0.8, 0.7), label='Lake'),
        Patch(facecolor=(0, 0.3, 0.6, 0.7), label='River')
    ]
    plt.legend(handles=legend_elements, loc='lower right')
    
    plt.axis('off')
    plt.title('Water Classification')
    plt.tight_layout()
    
    # Save
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

def extract_water_boundaries(classification, output_path, class_id=1, min_area=1000):
        """
        Extract water boundaries from the classification
        
        Parameters:
        classification: Classification array
        output_path: Path to save the vector file
        class_id: ID of the class to extract (1 for lakes, 2 for rivers)
        min_area: Minimum area in pixels to keep a polygon
        """
        # Create a binary mask for the target class
        binary_mask = (classification == class_id).astype(np.uint8)
        
        # Extract shapes
        shapes_results = []
        for shape, value in rasterio.features.shapes(binary_mask, mask=binary_mask > 0):
            shapes_results.append({
                'geometry': shape,
                'properties': {'class': class_id}
            })
        
        # Convert to GeoDataFrame
        if len(shapes_results) > 0:
            gdf = gpd.GeoDataFrame.from_features(shapes_results)
            
            # Calculate area
            gdf['area'] = gdf.geometry.area
            
            # Filter by minimum area
            gdf = gdf[gdf.area >= min_area]
            
            # Save to file if we have polygons left
            if len(gdf) > 0:
                gdf.to_file(output_path)
                logger.info(f"Extracted {len(gdf)} polygons to {output_path}")
            else:
                logger.warning(f"No polygons exceeding minimum area of {min_area} found")
        else:
            logger.warning(f"No polygons of class {class_id} found")

def main(band_paths_list, shapefile_path, output_dir, class_field='land_type', model_type='rf', 
         max_samples_per_polygon=10000, max_training_samples=500000, all_touched=True, min_valid_ratio=0.1):
    """
    Main function to run the water classification workflow
    
    Parameters:
    band_paths_list: List of dictionaries, each with paths to blue, green, red, nir bands
    shapefile_path: Path to the shapefile with polygon classifications
    output_dir: Directory to save outputs
    class_field: Field in the shapefile that contains the class labels
    model_type: 'rf' for Random Forest or 'svm' for SVM
    max_samples_per_polygon: Maximum number of samples to extract from each polygon
    max_training_samples: Maximum number of samples to use for model training
    all_touched: Whether to include all pixels touched by polygons
    min_valid_ratio: Minimum ratio of valid pixels needed for a polygon to be included
    """
    start_time = time.time()
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Configure logging to file
    log_file = os.path.join(output_dir, "processing.log")
    file_handler = logging.FileHandler(log_file)
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logger.addHandler(file_handler)
    
    logger.info("Starting water classification workflow")
    
    # Validate input files
    logger.info("Validating input files...")
    
    # Check shapefile
    if not os.path.exists(shapefile_path):
        error_msg = f"Shapefile not found: {shapefile_path}"
        logger.error(error_msg)
        raise FileNotFoundError(error_msg)
    
    # Check band files
    for i, band_paths in enumerate(band_paths_list):
        for band_name, band_path in band_paths.items():
            if not os.path.exists(band_path):
                error_msg = f"Band file not found: {band_path}"
                logger.error(error_msg)
                raise FileNotFoundError(error_msg)
    
    # Load polygon data
    logger.info("Loading polygon data...")
    try:
        polygons_gdf = load_polygons(shapefile_path, class_field)
    except Exception as e:
        logger.error(f"Error loading polygons: {e}")
        raise
    
    # Create training dataset from all images
    logger.info("Creating training dataset from multiple images...")
    try:
        X, y, polygon_ids, feature_names = create_training_dataset(
            band_paths_list, 
            polygons_gdf, 
            max_samples_per_polygon=max_samples_per_polygon,
            all_touched=all_touched,
            min_valid_ratio=min_valid_ratio
        )
    except Exception as e:
        logger.error(f"Error creating training dataset: {e}")
        raise
    
    # Report dataset size
    logger.info(f"Dataset size: {X.shape[0]} samples with {X.shape[1]} features")
    
    # Train model
    logger.info(f"Training {model_type} model...")
    model_results = {}
    
    if model_type in ['rf', 'both']:
        # Train Random Forest model
        logger.info("Training Random Forest model...")
        rf_results = train_water_model(X, y, polygon_ids, 'rf', max_training_samples)
        
        print("\nRandom Forest Results:")
        print(rf_results['report'])
        
        # Store model and results
        model_results['rf'] = rf_results
        
        # Save feature importance to CSV
        if rf_results['feature_importance'] is not None:
            # Get lengths for debugging
            feat_names_len = len(feature_names)
            feat_imp_len = len(rf_results['feature_importance'])
            print(f"Debug: feature_names length = {feat_names_len}, feature_importance length = {feat_imp_len}")

            # Option 1: Use only the common length
            min_length = min(feat_names_len, feat_imp_len)

            importance_df = pd.DataFrame({
                'Feature': feature_names[:min_length],
                'Importance': rf_results['feature_importance'][:min_length]
            })
    
            # Sort by importance (descending)
            importance_df = importance_df.sort_values('Importance', ascending=False)
            
            importance_path = os.path.join(output_dir, "feature_importance.csv")
            importance_df.to_csv(importance_path, index=False)
            logger.info(f"Feature importance saved to {importance_path}")
    
    if model_type in ['svm', 'both']:
        # Train SVM model with limited samples
        logger.info("Training SVM model...")
        # Use a smaller max_samples for SVM
        svm_max_samples = min(max_training_samples, 100000)  # Limit to 100K samples for SVM
        svm_results = train_water_model(X, y, 'svm', svm_max_samples)
        
        print("\nSVM Results:")
        print(svm_results['report'])
        
        # Store model and results
        model_results['svm'] = svm_results
    
    # Save the model(s)
    model_filename = os.path.join(output_dir, "water_classification_models.joblib")
    joblib.dump(model_results, model_filename)
    logger.info(f"Model(s) saved to {model_filename}")

    
    # Apply model to each image
    for i, band_paths in enumerate(band_paths_list):
        logger.info(f"Applying model to image {i+1}/{len(band_paths_list)}")
        
        # Generate output paths
        image_name = f"image_{i+1}"
        if 'blue' in band_paths:
            # Extract a recognizable name from the blue band path
            image_name = os.path.basename(band_paths['blue']).split('_')[0]
        
        output_classification = os.path.join(output_dir, f"{image_name}_classification.tif")
        
        # Apply the best model
        best_model_type = 'rf' if 'rf' in model_results else 'svm'
        classification = apply_model_to_image(
            model_results, 
            band_paths, 
            output_classification, 
            model_type=best_model_type,
            probabilities=True,
            batch_size=1000000
        )
        
        if classification is not None:
            # Extract water boundaries for lakes and rivers
            logger.info("Extracting water boundaries...")
            
            # Extract lake boundaries
            lake_output = os.path.join(output_dir, f"{image_name}_lake_boundaries.shp")
            extract_water_boundaries(classification, lake_output, class_id=1)
            
            # Extract river boundaries
            river_output = os.path.join(output_dir, f"{image_name}_river_boundaries.shp")
            extract_water_boundaries(classification, river_output, class_id=2)
    
    # Timing
    elapsed_time = time.time() - start_time
    hours, remainder = divmod(elapsed_time, 3600)
    minutes, seconds = divmod(remainder, 60)
    logger.info(f"Total processing time: {int(hours)}h {int(minutes)}m {int(seconds)}s")
    print(f"\nProcessing complete! Total time: {int(hours)}h {int(minutes)}m {int(seconds)}s")
    
    return model_results

if __name__ == "__main__":
    # Define paths to Sentinel-2 bands for each image
    # These are direct paths to each band JP2 file
    band_paths_list = [
        {
            # Image 1
            'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_1/T05VNJ_20230724T214539_B02_10m.tif",  # Blue band
            'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_1/T05VNJ_20230724T214539_B03_10m.tif", # Green band
            'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_1/T05VNJ_20230724T214539_B04_10m.tif",   # Red band
            'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_1/T05VNJ_20230724T214539_B08_10m.tif"   # NIR band
        },
        {
            # Image 2
            'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_2/T05VNK_20230721T213539_B02_10m.tif",
            'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_2/T05VNK_20230721T213539_B03_10m.tif",
            'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_2/T05VNK_20230721T213539_B04_10m.tif",
            'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_2/T05VNK_20230721T213539_B08_10m.tif"
        },
        {
            # Image 3
            'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_3/T05VPJ_20230723T212521_B02_10m.tif",
            'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_3/T05VPJ_20230723T212521_B03_10m.tif",
            'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_3/T05VPJ_20230723T212521_B04_10m.tif",
            'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_3/T05VPJ_20230723T212521_B08_10m.tif"
        },
        {
            # Image 4
            'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_4/T06VUQ_20230805T213531_B02_10m.tif",
            'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_4/T06VUQ_20230805T213531_B03_10m.tif",
            'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_4/T06VUQ_20230805T213531_B04_10m.tif",
            'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_4/T06VUQ_20230805T213531_B08_10m.tif"
        },
        {
            # Image 5
            'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_5/T06VWP_20230926T212529_B02_10m.tif",
            'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_5/T06VWP_20230926T212529_B03_10m.tif",
            'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_5/T06VWP_20230926T212529_B04_10m.tif",
            'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/GeoTIFF/image_5/T06VWP_20230926T212529_B08_10m.tif"
        }
        
    ]
    
    # Define path to your shapefile with polygon classifications
    polygons_path = "/Users/chloe/Documents/INDSTUDY/Training Data Images/Land_Type_Training/Training_b.shp"
    
    # Define output directory
    output_dir = "/Users/chloe/Documents/INDSTUDY/Training Data Images/Land_type_trainind/Output test classifications"
    
    # Run the workflow
    main(
        band_paths_list=band_paths_list,
        shapefile_path=polygons_path,
        output_dir=output_dir,
        class_field='land_type',           # Field in shapefile containing classes
        model_type='rf',                   # Use Random Forest only for speed
        max_samples_per_polygon=10000,     # Sample size per polygon
        max_training_samples=500000,       # Maximum samples for model training
        all_touched=True,                  # Include pixels that are partially covered by polygons
        min_valid_ratio=0.1                # Minimum ratio of valid pixels needed
    )

2025-05-02 11:35:50 - INFO - Starting water classification workflow
2025-05-02 11:35:50 - INFO - Validating input files...
2025-05-02 11:35:50 - INFO - Loading polygon data...
2025-05-02 11:35:50 - INFO - Creating training dataset from multiple images...


Shapefile columns: ['id', 'land_type', 'geometry']
Unique values in land_type: ['lake' 'land' 'river']
Shapefile CRS: EPSG:4326
Polygon class distribution:
  lake: 39 polygons
  land: 36 polygons
  river: 26 polygons

--- Processing image set 1/5 ---
Processing image set 1
Successfully loaded band from T05VNJ_20230724T214539_B02_10m.tif
Successfully loaded blue band from T05VNJ_20230724T214539_B02_10m.tif
Successfully loaded band from T05VNJ_20230724T214539_B03_10m.tif
Successfully loaded green band from T05VNJ_20230724T214539_B03_10m.tif
Successfully loaded band from T05VNJ_20230724T214539_B04_10m.tif
Successfully loaded red band from T05VNJ_20230724T214539_B04_10m.tif
Successfully loaded band from T05VNJ_20230724T214539_B08_10m.tif
Successfully loaded nir band from T05VNJ_20230724T214539_B08_10m.tif
Image bounds: 499980.0, 6790200.0, 609780.0, 6900000.0
Reprojecting polygons from EPSG:4326 to EPSG:32605
Found 27 overlapping polygons


Extracting from image set 1:   4%|▎         | 1/27 [00:00<00:20,  1.27it/s]

  Successfully extracted 10000 samples from polygon 32


Extracting from image set 1:   7%|▋         | 2/27 [00:01<00:13,  1.79it/s]

...

Successfully loaded band from T06VWP_20230926T212529_B02_10m.tif
Successfully loaded blue band from T06VWP_20230926T212529_B02_10m.tif
Successfully loaded band from T06VWP_20230926T212529_B03_10m.tif
Successfully loaded green band from T06VWP_20230926T212529_B03_10m.tif
Successfully loaded band from T06VWP_20230926T212529_B04_10m.tif
Successfully loaded red band from T06VWP_20230926T212529_B04_10m.tif
Successfully loaded band from T06VWP_20230926T212529_B08_10m.tif
Successfully loaded nir band from T06VWP_20230926T212529_B08_10m.tif


2025-05-02 11:45:12 - INFO - Valid pixels: 95540533 out of 120560400 (79.25%)


Prediction features: 4 (blue, green, red, nir)
After selection: 2 features


2025-05-02 11:46:44 - INFO - Extracting water boundaries...
/opt/anaconda3/envs/satellite/lib/python3.12/site-packages/pyogrio/geopandas.py:662: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
2025-05-02 11:46:53 - INFO - Created 579 records
2025-05-02 11:46:53 - INFO - Extracted 579 polygons to /Users/chloe/Documents/INDSTUDY/Training Data Images/Land_type_trainind/Output test classifications/T06VWP_lake_boundaries.shp
/opt/anaconda3/envs/satellite/lib/python3.12/site-packages/pyogrio/geopandas.py:662: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
2025-05-02 11:47:10 - INFO - Created 1,038 records
2025-05-02 11:47:10 - INFO - Extracted 1038 polygons to /Users/chloe/Documents/INDSTUDY/Training Data Images/Land_type_trainind/Output test classifications/T06VWP_river_boundaries.shp
20


Processing complete! Total time: 0h 11m 20s
